# Silver — espelho governado do bronze

A regra da casa, e ela não é negociável:

A silver é o espelho do bronze com governança aplicada. **Mesmo nome de tabela, mesmo grão, mesma contagem de linhas.**

## Permitido na silver

* **tipagem**: string vira `TIMESTAMP`, `INT`, `DATE`
* **legibilidade**: quebrar timestamp em data e hora
* **metadados**: `CURRENT` em toda coluna, tags na tabela
* **unificação**: dois cadastros do mesmo assunto, com a origem por registro
* **aritmética pura**: `atraso = real - previsto`

## Proibido na silver

* **filtro / `WHERE` de negócio** ❌
* **`GROUP BY` / agregação** ❌
* **limpar flag, classificação** ❌


**Por quê?** Porque a silver precisa servir várias análises, e toda linha que ela descarta é uma pergunta que ninguém mais vai conseguir fazer. Filtro fecha porta.

O teste para qualquer coluna nova: *isso embute uma decisão de negócio?* `atraso_partida_min = partida_real - partida_prevista` é subtração - silver. `partida_pontual = atraso <= 15>` embute o número **15**, que é decisão de negócio e  muda por cliente - gold.


# 1. O que precisa ser consertado na tipagem

Antes de escrever o `CAST`, medir. Duas armadilhas escondidas no bronze:

In [0]:
SELECT
    COUNT(*)                                                           AS linhas,
    SUM(CASE WHEN partida_real IS NULL THEN 1 ELSE 0 END)             AS partida_real_null_de_verdade,
    SUM(CASE WHEN partida_real LIKE 'null' THEN 1 ELSE 0 END)          AS partida_real_string_null,
    SUM(CASE WHEN partida_prevista = 'null' THEN 1 ELSE 0 END)        AS partida_prevista_string_null,
    SUM(CASE WHEN partida_prevista LIKE '%:%' THEN 1 ELSE 0 END)      AS com_fracao_de_segundo
FROM voebem.bronze.vra

**Armadilha 1 - a ausência veio como a string `null`.** Quatro caracteres de texto. `WHERE partida_real IS NULL` devolve **zero** numa tabela onde 29 mil voos não têm horário real. Correção: `nullif(coluna, 'null')` **antes** do cast.

**Armadilha 2 - dois formatos de timestamp no mesmo arquivo.** A maioria vem `2026-01-27 19:45:00`, mas ~80 mil linhas vêm com fração de segundo de 9 casas. Um `to_timestamp(col, 'yyyy-MM-dd HH:mm:ss')` fixo devolveria NULL para 8% da base, em silêncio. O `try_cast(... AS TIMESTAMP)` aceita os dois formatos, e o `try_` garante que um formato inválido retorna NULL em vez de explodir a query inteira.